[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C22_Reasoning_RL_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy 在**玩具环境**（可自动判对错的算术题、GridWorld）里**从零实现** o1/R1 背后的算法，再与**解析解或朴素实现对拍**。

这个 notebook 做四件事：① 确认环境；② 用一个最小例子体会「**可验证奖励**」长什么样；③ 建一个**可枚举**的玩具任务，从而能算出**真实答对概率**当 ground truth；④ 立下全课的纪律——**对拍（differential testing）**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画 TTC scaling / 帕累托曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 可验证奖励长什么样

推理 RL 的灵魂是**可验证奖励（verifiable reward）**：给一个问题和一个模型答案，用**程序**判对错，返回标量奖励。

和 RLHF 的「学一个奖励模型」不同，这里的奖励是 **ground-truth 验证**——几乎不可被风格性地 hack。下面写一个 GSM8K 风格的答案抽取+判分器（从 `#### 数字` 抽答案、与标准答案比对）。

In [ ]:
import re

def extract_answer(text):
    '''GSM8K 约定：最终答案写在 #### 之后。抽出末尾的数字。'''
    m = re.findall(r'####\s*(-?[0-9][0-9,]*)', text)
    if not m:
        return None
    return int(m[-1].replace(',', ''))

def outcome_reward(model_output, gold_answer):
    '''结果奖励 (ORM)：抽出的答案 == 标准答案 -> 1.0，否则 0.0。'''
    pred = extract_answer(model_output)
    return 1.0 if (pred is not None and pred == gold_answer) else 0.0

gold = 18
good = 'Janet sells 16-3-4=9 eggs ... 9*2=18 dollars.\n#### 18'
bad  = 'I think the answer is around twenty.\n#### 20'
none = 'lots of reasoning but forgot to give a final answer'
print('正确解   ->', outcome_reward(good, gold))
print('错误解   ->', outcome_reward(bad, gold))
print('无答案   ->', outcome_reward(none, gold))
assert outcome_reward(good, gold) == 1.0
assert outcome_reward(bad, gold) == 0.0
assert outcome_reward(none, gold) == 0.0
print('✅ 可验证奖励：对就是 1、错就是 0，程序说了算（不可花言巧语糊弄）')

## 3 · 一个可枚举的玩具任务：从此有了 ground truth

RL/PRM/搜索算法的对错很难直接看出来。我们的解法是用一个**小到可以枚举所有可能**的任务，从而能精确算出「真实答对概率」当裁判。

玩具任务 **凑数（make-target）**：给定数字集合，选一个动作序列让累加和命中目标 `T`。策略是一张 softmax 概率表。因为动作少，我们能**枚举所有轨迹**、算出任意策略的**精确成功率**——这是后面所有蒙特卡洛估计的对拍基准。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 玩具任务：3 步，每步从 {0,1,2,3} 选一个数，累加和 == TARGET 即成功。
ACTIONS = np.array([0, 1, 2, 3])
N_STEPS = 3
TARGET = 6

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def success(traj):
    '''可验证奖励：动作之和命中 TARGET -> 成功。'''
    return int(sum(ACTIONS[a] for a in traj) == TARGET)

def exact_success_prob(probs):
    '''枚举所有 |ACTIONS|^N_STEPS 条轨迹，按策略概率加权求精确成功率。
       probs[t] 是第 t 步的动作分布 (这里每步同分布以简化)。'''
    from itertools import product
    total = 0.0
    for traj in product(range(len(ACTIONS)), repeat=N_STEPS):
        ptraj = 1.0
        for t, a in enumerate(traj):
            ptraj *= probs[t][a]
        total += ptraj * success(traj)
    return total

# 均匀策略
logits = np.zeros((N_STEPS, len(ACTIONS)))
probs = softmax(logits)
p_exact = exact_success_prob(probs)
print(f'均匀策略的精确成功率 = {p_exact:.4f}')
# 蒙特卡洛估计应逼近它
N = 20000
wins = 0
for _ in range(N):
    traj = [rng.choice(len(ACTIONS), p=probs[t]) for t in range(N_STEPS)]
    wins += success(traj)
p_mc = wins / N
print(f'蒙特卡洛估计 ({N} 次)   = {p_mc:.4f}')
assert abs(p_mc - p_exact) < 0.02, 'MC 应逼近枚举的精确值'
print('✅ 有了可枚举的 ground truth：MC 估计逼近精确成功率，这是后面所有对拍的基石')

## 4 · 立纪律：对拍（differential testing）

本课每个算法都要和一个**可信参照**比对。最通用的参照之一是**数值梯度**（有限差分）：它慢但几乎不会写错，用来验证我们手写的**解析梯度**。

先把这个工作流跑通：对一个简单函数，对拍解析梯度 vs 有限差分。模块 01 的策略梯度就靠它兜底。

In [ ]:
def numerical_grad(f, x, eps=1e-6):
    '''中心差分估计 f 在 x 处的梯度（x 是 1D 向量）。'''
    g = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        xp = x.copy(); xp[i] += eps
        xm = x.copy(); xm[i] -= eps
        g[i] = (f(xp) - f(xm)) / (2 * eps)
    return g

# 例：f(x) = sum(softmax(x) * value)，解析梯度可推导
value = np.array([1.0, 0.0, -1.0, 2.0])
def f(x):
    return float(softmax(x) @ value)
def analytic_grad(x):
    s = softmax(x)
    # d/dx_i sum_j s_j v_j = s_i (v_i - sum_j s_j v_j)
    return s * (value - (s @ value))

x0 = np.array([0.3, -0.2, 0.5, 0.1])
g_num = numerical_grad(f, x0)
g_ana = analytic_grad(x0)
print('数值梯度 :', np.round(g_num, 5))
print('解析梯度 :', np.round(g_ana, 5))
assert np.allclose(g_num, g_ana, atol=1e-5), '解析梯度必须对拍数值梯度'
print('✅ 对拍通过：解析梯度 == 有限差分。这是验证策略梯度的统一裁判')

## 5 · 把「对拍」封成全课工具

把对拍封装成一个小函数，后面每个模块都用它判定「我的实现 == 可信参照」。它就是本课所有 `assert` 背后的统一裁判。

In [ ]:
def check_close(name, got, ref, atol=1e-8):
    '''对拍：实现结果 vs 可信参照（解析解/数值梯度/枚举值）。'''
    got_a, ref_a = np.asarray(got, dtype=float), np.asarray(ref, dtype=float)
    ok = np.allclose(got_a, ref_a, atol=atol)
    max_err = float(np.max(np.abs(got_a - ref_a))) if got_a.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参照不一致！'
    return ok

check_close('softmax 行和为 1', softmax(np.array([1.0, 2.0, 3.0])).sum(), 1.0)
check_close('MC vs 枚举成功率', p_mc, p_exact, atol=0.02)
check_close('解析梯度 vs 数值梯度', analytic_grad(x0), numerical_grad(f, x0), atol=1e-5)
print('\n这就是全课的工作流：写算法 -> 对拍可信参照 -> assert 兜底。')

## 6 · 预告：训练时算力 vs 测试时算力

推理模型有**两个**可投入算力的维度。用一个极简模型先建立直觉：假设单题准确率随「训练投入」与「测试时采样数」都上升，我们看同样的总预算怎么分。

In [ ]:
# 极简示意：训练投入 c_train 提升基础单样本正确率 p；测试时采 n 个样本 + 完美 verifier -> pass@n = 1-(1-p)^n
def base_acc(c_train):
    '''训练算力 -> 单样本正确率（示意：对数增长，封顶 0.9）。'''
    return 0.9 * (1 - np.exp(-c_train / 5.0))

def pass_at_n(p, n):
    '''完美 verifier 下 N 选优的覆盖率 = 至少一个对。'''
    return 1 - (1 - p) ** n

p_small = base_acc(2)    # 小训练投入
p_big   = base_acc(10)   # 大训练投入
print(f'小训练投入 单样本正确率 = {p_small:.3f}')
print(f'大训练投入 单样本正确率 = {p_big:.3f}')
print(f'小模型 + 8 次采样 pass@8 = {pass_at_n(p_small, 8):.3f}')
print(f'大模型 + 1 次采样 pass@1 = {pass_at_n(p_big, 1):.3f}')
assert pass_at_n(p_small, 8) > p_small, '测试时多采样应提升覆盖率'
assert pass_at_n(0.3, 10) > pass_at_n(0.3, 1)
print('\n✅ 关键直觉：训练时算力和测试时算力可以互相替代 —— 这是模块 05 的主题')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个 RL/PRM/搜索/TTC 算法，都会用 `np.allclose` 对拍解析解或数值梯度或枚举真值；结构正确则数值一致，数值一致则逻辑可迁移到 trl/verl。

**接下来五个模块**：01 Long-CoT 结果奖励 → 02 GRPO → 03 过程奖励 PRM → 04 verifier 搜索 → 05 TTC scaling。前三个讲训练时把推理塞进权重，后两个讲推理时把能力榨出来。

下一站：**模块 01 · Long-CoT 与结果奖励**。